# Train an IGNODE-compatible detector with YOLOX

**Output:** an ONNX model + sidecar JSON files ready for **Custom Model Upload** in your IGNODE workspace. The model is byte-equivalent to what IGNODE's in-platform detection trainer produces, so swapping later is a fresh-train job, not a sidecar rewrite.

This notebook mirrors the recipe in `ignode-trainer/benchmarks/yolox_upstream_baseline/train_any.py` — the same code IGNODE's production trainer (IR-3.O Path B) runs.

## Steps
1. Set `DATASET_URL` (public download link) or `DATASET_DIR` (Google Drive folder) in the Settings cell
2. Auto-normalize ANY format (VOC, COCO, YOLOv5/v8, Roboflow exports) → YOLOX's required VOC layout
3. Train
4. Export to ONNX (`decode_in_inference=False` — matches IGNODE UINF's `yolox_raw` decoder)
5. Write the 3 sidecar files (`preprocess_config.json`, `class_labels.json`, optional `manifest.json`)
6. Upload to IGNODE via Custom Model Upload

In [ ]:
# Step 0 — install YOLOX + dependencies. Cell ~3 min on a fresh runtime.
#
# IR-3.1.A.5 — Clone YOLOX from GitHub to /app/YOLOX so train_any.py's
# hardcoded `cwd="/app/YOLOX"` (matching the production trainer Docker
# image layout) works on Colab too. Then `pip install -e` makes the same
# clone importable as the `yolox` package — so tools/export_onnx.py in
# Step 4 is reachable at /app/YOLOX/tools/export_onnx.py, no install-dir
# guessing.
!mkdir -p /app && git clone --depth 1 --branch 0.3.0 https://github.com/Megvii-BaseDetection/YOLOX.git /app/YOLOX
!pip install -q -e /app/YOLOX --no-deps
!pip install -q supervision==0.21.0 onnx==1.21.0 onnxruntime==1.23.2 'pycocotools>=2.0.8' loguru tabulate ninja
!apt-get install -y -q g++ python3-dev


In [ ]:
# IR-3.S.B — Settings: pick your dataset source + tune training knobs.
# Set ONE of DATASET_URL or DATASET_DIR. URL takes precedence.
# Leave both empty and the next cell will warn + stop.
#
# Supported formats (auto-detected): voc, coco, yolov5/v8, roboflow exports.

# IR-3.S.F — Public download URL (.zip / .tar of the dataset).
#
# Quick test — copy this BCCD blood-cell-count sample (7.4 MB, 3 classes):
#   DATASET_URL = 'https://raw.githubusercontent.com/IGNODE-CONNECT/ignode-collab/main/examples/datasets/bccd.coco.zip'
#
# Your own data: any public .zip / .tar download link works
# (Roboflow Universe 'Raw URL', GitHub Release asset, public S3 / Drive).
DATASET_URL = ''

# Google Drive folder path. Mount Drive separately if you want this.
# Example: '/content/drive/MyDrive/my-detection-dataset'
DATASET_DIR = ''

# IR-3.1.A.6 — Training knobs in the same Settings cell as DATASET_URL
# so the customer edits ONE cell to tune their run.
# Reference: IR-3.O Path B production trainer defaults.
EPOCHS = 300         # 30 for a quick smoke; 300 for a real run
BATCH_SIZE = 16      # drop to 8 on a smaller GPU (T4 free Colab)

# Local scratch directory inside Colab — keep as-is.
WORKDIR = '/content/yolox-run'


In [ ]:
# IR-3.S.D — OPTIONAL Drive mount, gated so Run-all is safe.
# Only mounts Drive when the Settings cell points DATASET_DIR at a
# /content/drive path. URL customers + local-path customers skip
# this cell silently (no Drive auth popup, no Run-all stall).
import os

_needs_drive = (not DATASET_URL) and DATASET_DIR.startswith('/content/drive')
_already_mounted = os.path.ismount('/content/drive') or os.path.isdir('/content/drive/MyDrive')

if _needs_drive and not _already_mounted:
    from google.colab import drive
    drive.mount('/content/drive')
elif _needs_drive:
    print('Drive already mounted — skipping.')
else:
    print('Not using Drive (DATASET_URL set, or DATASET_DIR is a local path).')
    print('Skipping Drive mount.')


In [ ]:
# IR-3.S.B — Load dataset from whichever source the Settings cell set.
# Bails out loudly if BOTH DATASET_URL and DATASET_DIR are empty so the
# customer doesn't waste a 30-minute training run on an empty folder.
import os, pathlib, shutil, zipfile, tarfile, subprocess

if not DATASET_URL and not DATASET_DIR:
    raise SystemExit(
        '\n'
        '⚠️  Both DATASET_URL and DATASET_DIR are empty in the Settings cell.\n'
        '   Set ONE of them before running this cell:\n'
        '     • DATASET_URL — a public .zip / .tar download link\n'
        '     • DATASET_DIR — a path inside your mounted Google Drive\n'
        '   Then re-run this cell.'
    )

pathlib.Path(WORKDIR).mkdir(parents=True, exist_ok=True)

if DATASET_URL:
    print(f'Downloading from public URL: {DATASET_URL}')
    _archive = pathlib.Path('/content/_dataset_download')
    _archive.mkdir(parents=True, exist_ok=True)
    _dl_path = _archive / 'dataset.zip'
    subprocess.run(['curl', '-fsSL', '-o', str(_dl_path), DATASET_URL], check=True)
    print(f'Downloaded {_dl_path.stat().st_size:,} bytes — extracting…')
    _extracted = _archive / 'unpacked'
    if _extracted.exists():
        shutil.rmtree(_extracted)
    _extracted.mkdir()
    if zipfile.is_zipfile(_dl_path):
        with zipfile.ZipFile(_dl_path) as zf:
            zf.extractall(_extracted)
    elif tarfile.is_tarfile(_dl_path):
        with tarfile.open(_dl_path) as tf:
            tf.extractall(_extracted)
    else:
        raise SystemExit(f'Downloaded file is neither .zip nor .tar: {_dl_path}')
    DATASET_DIR = str(_extracted)
    print(f'Extracted to: {DATASET_DIR}')
else:
    if not pathlib.Path(DATASET_DIR).is_dir():
        raise SystemExit(
            f'❌ DATASET_DIR={DATASET_DIR!r} does not exist or is not a directory.\n'
            f'   If this is a Google Drive path, run the Drive mount cell below\n'
            f'   first — or set DATASET_URL instead.'
        )
    print(f'Using local/Drive folder: {DATASET_DIR}')

!ls -la "{DATASET_DIR}" | head -20


In [ ]:
# Step 2 — download train_any.py + helpers from the public ignode-collab
# GitHub mirror. (Canonical source is ignode-trainer/benchmarks/
# yolox_upstream_baseline/ but that lives on private Bitbucket — the
# Colab kernel can't authenticate, so we mirror to ignode-collab and
# the customer Colab pulls from there.)
# Same code IGNODE's production trainer (IR-3.O Path B) runs internally.
BENCHMARK_RAW = 'https://raw.githubusercontent.com/IGNODE-CONNECT/ignode-collab/main/colab/yolox'
!curl -fsSL {BENCHMARK_RAW}/train_any.py -o {WORKDIR}/train_any.py
!curl -fsSL {BENCHMARK_RAW}/prepare_dataset.py -o {WORKDIR}/prepare_dataset.py
!curl -fsSL {BENCHMARK_RAW}/voc_eval_patch.py -o {WORKDIR}/voc_eval_patch.py
!ls -la {WORKDIR}


In [ ]:
# Step 3 — train. Defaults match the IR-3.O production recipe:
#   --epochs 300, --batch-size 16, --backbone yolox_s, --fp16, --image-size 640
# Settings cell has EPOCHS + BATCH_SIZE — edit there, not here.
import os
os.chdir(WORKDIR)
!python train_any.py     --data {DATASET_DIR}     --output {WORKDIR}/run_001     --epochs {EPOCHS}     --batch-size {BATCH_SIZE}     --backbone yolox_s     --image-size 640     --fp16


In [ ]:
# Step 4 — export to ONNX matching IGNODE's UINF yolox_raw contract.
# Critical flag: --decode_in_inference is OMITTED. UINF host-side decodes via
# anchor-grid + stride projection, which requires raw network outputs at
# strides 8/16/32. IR-3.Q catalogs this contract.
#
# IR-3.1.A.5 — Use the YOLOX clone Cell 1 placed at /app/YOLOX (matches
# train_any.py's hardcoded layout). The exp file there was patched
# in-place during training (train_any.py line 346), so it carries the
# correct num_classes / depth / width for THIS run.
import os
CKPT = f'{WORKDIR}/run_001/yolox_voc_s/best_ckpt.pth'
OUT  = f'{WORKDIR}/run_001/model.onnx'
EXP  = '/app/YOLOX/exps/example/yolox_voc/yolox_voc_s.py'

os.chdir('/app/YOLOX')
!python tools/export_onnx.py -f {EXP} -c {CKPT} --output-name {OUT} --opset 18


In [ ]:
# IR-3.S.A — Step 5: write sidecars + auto-derive CLASS_LABELS.
# DO NOT hand-type class names here. train_any.py wrote a classes.txt
# during normalization that encodes EXACTLY the index→name mapping
# YOLOX trained against. Typing them in a different order ships a
# model where every prediction's label is rotated (the IR-3.Y trap).
# This cell reads classes.txt directly; the next cell smoke-tests
# the exported ONNX so you SEE the mapping before uploading.
#
# Values MUST match the production trainer's sidecar shape
# (see IR-3.BB + IR-3.CC). The 4 fields that matter most:
#   channel_order: 'BGR'    — Megvii uses cv2.imread
#   rescale:       'none'   — YOLOX trains on raw [0, 255]
#   mean / std:    identity — no normalization
#   postprocess.family: 'yolox_raw' — UINF host-side decode
import json, shutil, pathlib

# 1. Find the classes.txt train_any.py produced. Locations vary
# slightly by dataset format — search the run dir.
_run = pathlib.Path(f'{WORKDIR}/run_001')
_candidates = sorted(_run.rglob('classes.txt'))
if not _candidates:
    raise SystemExit(
        'No classes.txt found under ' + str(_run) + ' — train_any.py probably failed '
        'to normalize your dataset. Re-run Step 3 + inspect its logs before continuing.'
    )
_classes_txt = _candidates[0]
CLASS_LABELS = [ln.strip() for ln in _classes_txt.read_text().splitlines() if ln.strip()]
print(f'Derived CLASS_LABELS from {_classes_txt}:')
for i, name in enumerate(CLASS_LABELS):
    print(f'  [{i}] {name}')
print()
print('If this order surprises you, STOP and check your source dataset.')
print('Wrong order here = every prediction is mislabeled at deploy time.')

preprocess_config = {
    'input_size':      [640, 640],
    'mean':            [0.0, 0.0, 0.0],
    'std':             [1.0, 1.0, 1.0],
    'channel_order':   'BGR',
    'image_format':    'CHW',
    'rescale':         'none',
    'resize_method':   'letterbox',
    'letterbox_color': [114, 114, 114],
    'postprocess': {
        'family':               'yolox_raw',
        'nms_required':         True,
        'confidence_threshold': 0.25,
        'nms_iou_threshold':    0.65,
    },
    '_backbone': 'yolox_s',
}

out = pathlib.Path(f'{WORKDIR}/run_001/upload-bundle')
out.mkdir(exist_ok=True)
shutil.copy(f'{WORKDIR}/run_001/model.onnx', out / 'model.onnx')
(out / 'preprocess_config.json').write_text(json.dumps(preprocess_config, indent=2))
(out / 'class_labels.json').write_text(json.dumps(CLASS_LABELS, indent=2))

print()
print('Upload bundle ready at:', out)
!ls -la {out}


In [ ]:
# IR-3.S.A — Step 5.5: ONNX smoke-test (DO NOT SKIP).
# Runs the freshly-exported YOLOX-raw ONNX on ONE image and prints
# the highest-objectness anchor's argmax class index + its label.
# If the top prediction is labeled in a way that obviously contradicts
# the image's content, classes.txt got reordered somewhere upstream —
# fix it BEFORE uploading.
import json, pathlib, numpy as np, onnxruntime as ort
import cv2  # Megvii reads BGR via cv2; mirror that here.

_bundle = pathlib.Path(f'{WORKDIR}/run_001/upload-bundle')
_labels = json.loads((_bundle / 'class_labels.json').read_text())
_sess   = ort.InferenceSession(str(_bundle / 'model.onnx'), providers=['CPUExecutionProvider'])
_inp    = _sess.get_inputs()[0].name

# Pick the first val image. train_any.py keeps val under run_001/.
# Tweak this search if your layout differs.
_run = pathlib.Path(f'{WORKDIR}/run_001')
_imgs = [p for p in _run.rglob('*.jpg') if 'val' in p.parts or 'test' in p.parts]
if not _imgs:
    _imgs = sorted(_run.rglob('*.jpg'))[:1]
assert _imgs, 'No JPGs under ' + str(_run) + ' — adjust the smoke-test search.'
_img_path = _imgs[0]
print(f'Smoke-testing on: {_img_path.relative_to(_run)}')

# YOLOX preprocess: BGR, letterbox to 640×640, [0..255], CHW, no normalization.
_bgr = cv2.imread(str(_img_path))
_h, _w = _bgr.shape[:2]
_scale = min(640 / _h, 640 / _w)
_nh, _nw = int(_h * _scale), int(_w * _scale)
_resized = cv2.resize(_bgr, (_nw, _nh), interpolation=cv2.INTER_LINEAR)
_canvas = np.full((640, 640, 3), 114, dtype=np.uint8)
_canvas[:_nh, :_nw] = _resized
_arr = _canvas.astype(np.float32).transpose(2, 0, 1)[None, ...]

# YOLOX-raw output: (1, N_anchors, 5 + num_classes). 5 = [cx, cy, w, h, obj].
_outs = _sess.run(None, {_inp: _arr})
_raw  = next((o for o in _outs if o.ndim == 3 and o.shape[-1] >= 5 + len(_labels)), None)
assert _raw is not None, 'Could not find a YOLOX-raw-shaped output — confirm Step 4 export.'
_obj   = 1.0 / (1.0 + np.exp(-_raw[0, :, 4]))           # sigmoid objectness
_cls_l = 1.0 / (1.0 + np.exp(-_raw[0, :, 5:5 + len(_labels)]))  # per-class sigmoid
_best  = int((_obj[:, None] * _cls_l).max(axis=1).argmax())
_best_class = int(_cls_l[_best].argmax())
_best_obj   = float(_obj[_best])
_best_score = float((_obj[_best] * _cls_l[_best]).max())
print()
print(f'Top anchor: idx={_best_class} → class_labels[{_best_class}] = {_labels[_best_class]!r}')
print(f'Objectness: {_best_obj:.3f}  Composite score: {_best_score:.3f}')
print()
print(f'Sanity check: open {_img_path.name}. If the model thinks it contains')
print(f'a {_labels[_best_class]!r} but you can see it does not, STOP and fix')
print('upload-bundle/class_labels.json before uploading.')


## Step 6 — upload to IGNODE

In your IGNODE workspace:
1. ML Factory → Models → **Upload custom model**
2. Drag the 3 files from `upload-bundle/` into the modal
3. Deploy to a UINF instance
4. Verify in Playground — boxes should land tight on your test image

If the boxes are mispositioned: usually a channel-order or class-label-order issue. The benchmark code on Vultr at `/root/applications/ignode-trainer-git/benchmarks/yolox_upstream_baseline` is the reference recipe; any deviation in the sidecar JSON is the place to debug.